# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 5, (2, 3). Потому что на первой строке и на втором столбце находится число 5, и shape выведет форму листа, тоесть (2(строки), 3(столбца))
2. 1, 8. Потому что argmax() возвращает индекс максимального элемента, их двое, поэтому берется индекс первого элемента, а также max() возвращает максимальное значение
3. 99. Потому что у a и b используют то же место в памяти, и когда число под индексом 0 меняется в переменной b, то и в a происходит изменение на то же число.
4. 2, 0.5. Потому что a > 3 проверяет каждый элемент массива, и если условие верно, то он возвращает значение True(1) или False(0), и затем суммирует все и выходит 2, а также mean() выводит среднее значение элементов в листе, тоесть 2/4 = 0.5
5. (2, 3), 0.25. Потому что shape выводит форму листа, а linspace() делит элементы равномерно по указанному числу, и в конце [1] выводит элемент с индексом 1

In [4]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders.csv
orders.csv на месте: True


In [5]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

(5463,) float64 bool


## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [10]:
# База

In [15]:
#1
print(f"Индекс: {amount.argmax()}\nСумма: {amount[amount.argmax()]}\nКлиент: {cust[amount.argmax()]}\nГород: {city[amount.argmax()]}")

Индекс: 1001
Сумма: 9999990.0
Клиент: TEST
Город: Алматы


In [17]:
#2
print(f"Выручка оплаченных заказов: {amount[paid].sum()}\nВыручка неоплаченных заказов: {amount[~paid].sum()}")

Выручка оплаченных заказов: 101909360.0
Выручка неоплаченных заказов: 7974000.0


In [18]:
#3
print(f"Количество заказов в Шымкенте: {city[city == 'Шымкент'].shape[0]}\nДоля заказов в Шымкенте: {city[city == 'Шымкент'].shape[0] / city.shape[0]}")

Количество заказов в Шымкенте: 804
Доля заказов в Шымкенте: 0.1471718835804503


## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [ ]:
# Стандарт

In [23]:
#4
mask = paid & (amount > 50_000)
loss_all = amount[mask].sum() * 0.15
loss_without_test = amount[mask & (cust != "TEST")].sum() * 0.15

print(f"Со всеми заказами: {loss_all:,.2f} ₸")
print(f"Без клиента test: {loss_without_test:,.2f} ₸")

Со всеми заказами: 5,702,422.50 ₸
Без клиента test: 1,202,427.00 ₸


ответы сильно различаются потому что у test есть 3 неоплаченных заказа по 9 999 990, и все они получают скидки и добавляются к потерям

In [25]:
#5
for name in np.unique(city):
    city_mask = city == name
    fast = (days[city_mask] >= 1) & (days[city_mask] <= 2)
    print(f"{name}: {fast.mean():.1%}")

Актобе: 62.2%
Алматы: 64.5%
Астана: 63.5%
Атырау: 64.4%
Караганда: 67.8%
Шымкент: 29.9%


In [26]:
#6
mask = paid & (cust != "TEST")
avg = amount[mask].mean()
print(f"Средний чек: {avg:.2f} ₸")

Средний чек: 14672.39 ₸


1. amount.mean() учитывает также неоплаченные заказы, а нам нужны оплаченные
2. учитываются тестовые заказы которые сильно завышают средний чек, их нужно убрать

## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# Челлендж